# Speaker-attributed transcription — WhisperX (Colab GPU)

Diaryze + transcribe a multi-speaker recording into a **timestamped transcript with named speakers**.

**Pipeline (WhisperX, all open-source)**
- `faster-whisper` large-v3 → ASR (top WER band on Artificial Analysis STT arena)
- wav2vec2 **forced alignment** → tight word timestamps (the accuracy lever for word→speaker)
- `pyannote/speaker-diarization-community-1` → speaker turns (WhisperX's default; ~14.7% DER on DIHARD-III, best open-source)
- `assign_word_speakers` (interval-tree overlap) → words tagged by speaker
- semi-automatic cluster → name mapping (self-intros + host handoff) → human-verify cell

**Layout**
1. **CONFIG** — the only cell you edit per source.
2. **Setup** — run once: GPU check, install, imports, credentials, model-weight downloads.
3. **Run** — acquire audio → resample → transcribe+align → diarize+assign → name-map → render → download.

**Before running**
1. **Runtime → Change runtime type → T4 GPU** (L4/A100 faster).
2. Accept the license on the [`pyannote/speaker-diarization-community-1`](https://huggingface.co/pyannote/speaker-diarization-community-1) model card.
3. Add a Colab secret **`HF_TOKEN`** (HuggingFace *read* token): left sidebar → 🔑 Secrets. Read via `userdata`; never printed.
   - If a model load fails with 401/403 on some `pyannote/...` repo, accept that sub-model's license too (the error names the URL).

In [ ]:
# ===== CONFIG — edit per source =====
# Speakers in a sensible order; host first helps the handoff heuristic.
SPEAKERS = [
    {"label": "Capehart", "names": ["Jonathan Capehart", "Jonathan", "Capehart"]},
    {"label": "Sinzdak",  "names": ["Colleen Roh Sinzdak", "Colleen", "Sinzdak"]},
    {"label": "Murray",   "names": ["Melissa Murray", "Melissa", "Murray"]},
    {"label": "Isgur",    "names": ["Sarah Isgur", "Sarah", "Isgur"]},
]

MIN_SPEAKERS = 4          # leave headroom for an announcer / audience Q&A
MAX_SPEAKERS = 6

WHISPER_MODEL = "large-v3"   # or "large-v3-turbo" (faster, marginally weaker)
COMPUTE_TYPE  = "float16"    # "int8_float16" if VRAM-tight
BATCH_SIZE    = 16

AUDIO_INPUT = None           # None -> upload/Drive in the Run block. e.g. "/content/drive/MyDrive/.../panel.m4a"
OUTPUT_NAME  = "sc-2025-term-panel"
SPEAKER_MAP_OVERRIDE = None  # prefill to skip auto name-mapping

---
## Setup — run once

GPU check, install, imports, credentials, and model-weight downloads. Everything heavy lives here so the Run block is pure compute.

**Heads up — one-time runtime restart:** the install cell pins a coherent `torch` / `torchvision` / `torchaudio` triple. Left to itself, Colab ends up with torchvision built for the *old* torch and you hit `operator torchvision::nms does not exist` on import. If Colab prompts **"Restart session"** after the install cell, click it, then re-run that cell (it detects the good stack and skips) and continue.

In [ ]:
# GPU + ffmpeg + install a coherent torch / WhisperX stack
import subprocess, sys, torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
print("ffmpeg:", subprocess.run(["ffmpeg","-version"], capture_output=True, text=True).stdout.splitlines()[0])

# WhisperX pins torch~=2.8. On Colab that can leave torchvision built for the
# OLD torch -> 'operator torchvision::nms does not exist' on import. Install the
# whole torch/torchvision/torchaudio triple pinned together so the C++ ABI
# matches, then WhisperX. Skip if the stack is already coherent.
def _stack_ok():
    try:
        import whisperx
        import torchvision  # reproduces the nms ABI error if skewed
        return torch.__version__.startswith("2.8")
    except Exception:
        return False

if not _stack_ok():
    print("installing torch 2.8.0 + torchvision 0.23.0 + torchaudio 2.8.0 + whisperx ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0", "whisperx"])
    print("\nDone. If Colab shows a 'Restart session' prompt (torch was upgraded),\n"
          "click it, then re-run THIS cell (it will skip) and continue below.")
else:
    print("torch 2.8 + whisperx ready (ABI coherent).")

In [ ]:
# Imports + HuggingFace credential
import os, re, json, subprocess, datetime
from collections import defaultdict
import whisperx
from whisperx.diarize import DiarizationPipeline
from google.colab import userdata, drive, files

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if not HF_TOKEN or len(HF_TOKEN) < 10:
    raise SystemExit("No HF_TOKEN secret. Add a Colab secret named HF_TOKEN "
                     "(HuggingFace read token) and accept the community-1 license.")
print("imports OK; HF token loaded (not printed).")

In [ ]:
# Load all models (downloads weights on first run)
print("loading ASR model ...")
asr_model = whisperx.load_model(WHISPER_MODEL, device="cuda", compute_type=COMPUTE_TYPE)
print("loading alignment model (wav2vec2) ...")
align_model, align_meta = whisperx.load_align_model("en", device="cuda")
print("loading diarization pipeline (pyannote community-1) ...")
diarize_pipeline = DiarizationPipeline(token=HF_TOKEN, device="cuda")
print("all models loaded.")

---
## Run

Acquire the file, process it, export the transcript. Re-run from here for a new source (models stay loaded).

In [ ]:
# Acquire the audio file
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True)
src = AUDIO_INPUT
if src and os.path.exists(src):
    audio_path = src
    print(f"Using configured audio: {audio_path}")
else:
    if input("Mount Google Drive to read the file? (y/N): ").strip().lower() == "y":
        drive.mount("/content/drive")
        audio_path = input("Drive path to audio (e.g. /content/drive/MyDrive/.../panel.m4a): ").strip().strip('"')
    else:
        print("Pick the file in the upload dialog...")
        up = files.upload()
        if not up:
            raise SystemExit("No file uploaded.")
        fn = list(up)[0]
        audio_path = os.path.join(WORK, fn)
        import shutil; shutil.move(os.path.join("/content", fn), audio_path)
assert os.path.exists(audio_path), f"File not found: {audio_path}"
print(f"audio: {audio_path} ({os.path.getsize(audio_path)/1e6:.1f} MB)")

In [ ]:
# Resample to 16kHz mono wav (what both pipelines expect)
WAV16 = os.path.join(WORK, "audio_16k_mono.wav")
subprocess.run(["ffmpeg","-y","-i", audio_path, "-ac","1","-ar","16000",
                "-c:a","pcm_s16le", WAV16], check=True, capture_output=True)
print("16kHz mono:", WAV16)

In [ ]:
# Transcribe (batched, built-in VAD) + force-align words
audio = whisperx.load_audio(WAV16)
print("transcribing ...")
result = asr_model.transcribe(audio, batch_size=BATCH_SIZE, language="en")
print(f"{len(result['segments'])} segments, language={result['language']}")
print("force-aligning word timestamps ...")
result = whisperx.align(result["segments"], align_model, align_meta, audio, device="cuda")
print(f"{sum(len(s.get('words', [])) for s in result['segments'])} aligned words")

In [ ]:
# Diarize (community-1) + assign words -> speakers
print("diarizing ...")
diarize_df = diarize_pipeline(audio, min_speakers=MIN_SPEAKERS, max_speakers=MAX_SPEAKERS)
result = whisperx.assign_word_speakers(diarize_df, result, fill_nearest=True)

labels = sorted(diarize_df["speaker"].unique())
print(f"{len(diarize_df)} turns, {len(labels)} speakers: {labels}")

words = []
for seg in result["segments"]:
    for w in seg.get("words", []):
        s = w.get("start"); e = w.get("end", s)
        if s is None:
            continue
        words.append({"start": float(s), "end": float(e), "text": w.get("word",""), "speaker": w.get("speaker")})
print(f"{sum(1 for w in words if w['speaker'])}/{len(words)} words labeled")

In [ ]:
# Propose cluster -> name (VERIFY against the printed evidence)
if SPEAKER_MAP_OVERRIDE:
    SPEAKER_MAP = dict(SPEAKER_MAP_OVERRIDE)
else:
    ctext, ctimes = {}, {}
    for w in words:
        ctext.setdefault(w["speaker"], []).append(w["text"])
        ctimes.setdefault(w["speaker"], []).append((w["start"], w["end"]))
    ctext = {k: " ".join(v) for k, v in ctext.items()}

    SPEAKER_MAP = {}
    intro_re = re.compile(r"\b(?:i['']m|i am|my name(?:'s| is)|this is)\s+([A-Za-z'\-]+)", re.I)

    # Pass 1: self-introductions ("I'm Jonathan Capehart").
    for sp in SPEAKERS:
        for name in sp["names"]:
            first, last = name.split()[0].lower(), name.split()[-1].lower()
            for clu, txt in ctext.items():
                if clu in SPEAKER_MAP:
                    continue
                if any(m.group(1).lower() in (first, last) for m in intro_re.finditer(txt)):
                    SPEAKER_MAP[clu] = sp["label"]

    # Pass 2: handoff - host names a guest; next non-host voice is that guest.
    host_cluster = next((c for c, l in SPEAKER_MAP.items() if l == SPEAKERS[0]["label"]), None)
    if host_cluster is None:
        host_cluster = min(ctimes, key=lambda c: ctimes[c][0][0])
    host_words = [w for w in words if w["speaker"] == host_cluster]
    timeline = sorted(words, key=lambda x: x["start"])
    for sp in SPEAKERS:
        if sp["label"] in SPEAKER_MAP.values():
            continue
        first = sp["names"][0].split()[0]
        t_name = next((w["start"] for w in host_words if first.lower() in w["text"].lower()), None)
        if t_name is None:
            continue
        for w in timeline:
            if (w["speaker"] != host_cluster and w["start"] >= t_name
                    and w["speaker"] not in SPEAKER_MAP):
                SPEAKER_MAP[w["speaker"]] = sp["label"]
                break

print("Proposed cluster -> name (VERIFY):")
for l in labels:
    print(f"  {l:14s} -> {SPEAKER_MAP.get(l, 'UNMAPPED')}")
print("\nEvidence - first 30 words each cluster actually said:")
for l in labels:
    ws = [w for w in words if w["speaker"] == l][:30]
    print(f"\n[{l}] " + "".join(w["text"] for w in ws))
print("\nIf anything is wrong/UNMAPPED, edit SPEAKER_MAP in the NEXT cell.")

In [ ]:
# ===== Edit SPEAKER_MAP here if the proposal was wrong, then run =====
# SPEAKER_MAP = {"SPEAKER_00":"Capehart","SPEAKER_01":"Murray","SPEAKER_02":"Isgur","SPEAKER_03":"Sinzdak"}

def name_of(l): return SPEAKER_MAP.get(l, l)
def fmt_t(s):
    s = max(0, int(round(s)))
    return f"{s//3600:02d}:{(s%3600)//60:02d}:{s%60:02d}"

# Merge words into turns: split on speaker change or gap > GAP_S
GAP_S = 1.0
turns_out, cur = [], None
for w in sorted(words, key=lambda x: x["start"]):
    if cur and cur["speaker"] == w["speaker"] and (w["start"] - cur["end"]) <= GAP_S:
        cur["end"] = w["end"]; cur["text"] += w["text"]
    else:
        if cur: turns_out.append(cur)
        cur = {"speaker": w["speaker"], "start": w["start"], "end": w["end"], "text": w["text"]}
if cur: turns_out.append(cur)

# markdown
mdbuf = [f"# {OUTPUT_NAME}", "",
         f"_Generated {datetime.datetime.utcnow().isoformat(timespec='seconds')}Z | "
         f"{len(turns_out)} turns | {len(labels)} speakers_", ""]
for t in turns_out:
    mdbuf.append(f"[{fmt_t(t['start'])}] **{name_of(t['speaker'])}**: {t['text'].strip()}")
with open(os.path.join(WORK, OUTPUT_NAME + ".md"), "w") as f:
    f.write("\n".join(mdbuf))

# json (structured turns - use this for wiki filing)
with open(os.path.join(WORK, OUTPUT_NAME + ".json"), "w") as f:
    json.dump([{"speaker": name_of(t["speaker"]), "start": t["start"],
                "end": t["end"], "text": t["text"].strip()} for t in turns_out], f, indent=2)

# txt
with open(os.path.join(WORK, OUTPUT_NAME + ".txt"), "w") as f:
    f.write("\n".join(f"[{fmt_t(t['start'])}] {name_of(t['speaker'])}: {t['text'].strip()}"
                      for t in turns_out))

talk = defaultdict(float)
for t in turns_out:
    talk[name_of(t["speaker"])] += t["end"] - t["start"]
print("Talk time per speaker:")
for sp, dur in sorted(talk.items(), key=lambda x: -x[1]):
    print(f"  {sp:20s} {dur/60:6.1f} min")
print(f"\nwrote {OUTPUT_NAME}.md / .json / .txt to {WORK}")

In [ ]:
# Download the outputs
for ext in ("md", "json", "txt"):
    p = os.path.join(WORK, f"{OUTPUT_NAME}.{ext}")
    if os.path.exists(p):
        files.download(p)
print("downloads triggered (files also under /content/work)")

## Next step

1. `.md` / `.txt` are the readable transcript; `.json` is structured turns (`speaker, start, end, text`) — use it for the wiki filing.
2. Drop the transcript into the wiki's `raw/` (as the source's transcript asset), then run the **wiki-filing** skill on it.
3. A lingering `SPEAKER_xx` (announcer / audience) is expected — file it as an unattributed voice or merge it in the filing pass.

For a new source: edit CONFIG, then re-run from the **Run** block (models stay loaded).